# PINN 入门教程
## Physics-Informed Neural Networks：物理信息神经网络

本教程从一阶常微分方程出发，介绍 PINN 的基本思路：用神经网络学习未知函数，并用物理方程约束函数形式。

### 适合人群

- 有 Python 基础，了解函数、数组和基本绘图；
- 学过基本微积分，了解导数和微分方程；
- 希望先通过简单例子理解 PINN 的基本流程。

### 学习目标

- 理解 PINN 与普通神经网络拟合的区别；
- 用神经网络表示未知函数；
- 用自动微分计算方程残差；
- 将物理方程、初始条件或边界条件写入损失函数；
- 训练模型，并与解析解比较。




---
## 准备工作：安装 Python 库

本教程使用三个库：

| 库 | 作用 | 安装命令 |
|------|------|------|
| **PyTorch** | 神经网络与自动微分 | `pip install torch` |
| **NumPy** | 数值计算与数组处理 | `pip install numpy` |
| **Matplotlib** | 绘图 | `pip install matplotlib` |

一次性安装命令为：

```bash
pip install torch numpy matplotlib
```

本例计算量较小，可直接使用 CPU 运行。安装后可用以下代码检查 PyTorch 是否可用：

```python
import torch
print(torch.__version__)
```




In [ ]:
# 安装依赖；已安装时可跳过
pip install torch numpy matplotlib



---
## 1. 什么是 PINN？

PINN 的全称是 **Physics-Informed Neural Networks**，通常译为“物理信息神经网络”。

普通神经网络主要从数据中学习输入与输出之间的关系。PINN 在此基础上加入物理方程约束，使网络预测结果同时满足数据、微分方程、初始条件或边界条件。

### 与传统数值方法的区别

| 传统数值方法，例如有限差分 | PINN |
|---|---|
| 先离散方程，再求离散点上的解 | 用神经网络表示连续函数 |
| 非网格点结果通常需要插值 | 任意输入点均可得到网络预测 |
| 网格设计和边界处理较关键 | 采样点设置相对灵活 |
| 标准正问题中通常更稳定、更高效 | 在反问题和数据稀疏问题中较有优势 |

PINN 不是传统数值方法的替代品。对于多数标准正问题，有限差分、有限元、有限体积等方法仍然更稳定、高效。PINN 的主要价值在于把观测数据和物理方程放入同一个优化框架，尤其适用于数据不完整、参数未知或反问题等情形。

### 损失函数中的权重系数 $\lambda$

PINN 的总损失通常由多个部分组成，例如：

$$
\mathcal{L}_{\text{total}} = \lambda_1 \mathcal{L}_{PDE} + \lambda_2 \mathcal{L}_{BC} + \cdots
$$

其中 $\mathcal{L}_{PDE}$ 表示方程残差损失，$\mathcal{L}_{BC}$ 表示边界条件或初始条件损失。

引入权重系数的原因是不同损失项的数值量级可能不同。若某一项量级过大，训练会主要优化该项，其他约束可能被弱化。

常见处理方式如下：

| 方法 | 做法 | 适用情况 |
|------|------|------|
| 固定权重 | 手动指定，例如本例取 $\lambda_{BC}=10$ | 简单问题或入门算例 |
| 等权重 | 所有权重取 1 | 各项损失量级接近 |
| 自适应权重 | 训练过程中动态调整 | 较复杂的问题 |

实际训练中，如果边界条件误差较大，可适当增大边界损失权重；如果边界已满足而方程残差较大，则应避免边界项过度主导训练。



---
## 2. 例子：用 PINN 求解一阶 ODE

考虑指数衰减方程：

$$
\frac{du}{dt} = -u, \quad t \in [0, 3].
$$

初始条件为：

$$
u(0) = 1.
$$

解析解为：

$$
u(t) = e^{-t}.
$$

该问题形式简单，且有解析解，适合作为 PINN 的入门算例。




In [ ]:
# 导入库
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# 固定随机种子，便于复现实验
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch 版本: {torch.__version__}")



---
## 3. 构建神经网络

用一个小型全连接神经网络表示未知函数 $u(t)$。

记网络输出为 $u_\theta(t)$，其中 $\theta$ 表示网络的全部可训练参数。训练目标是调整 $\theta$，使 $u_\theta(t)$ 满足微分方程和初始条件。

本例采用以下结构：

- 输入：时间 $t$；
- 隐藏层：3 层，每层 20 个神经元；
- 激活函数：$\tanh$；
- 输出：网络预测值 $u(t)$。

PINN 需要对网络输出求导，因此激活函数通常选用光滑函数。$\tanh$ 连续可微，是入门算例中常用的选择。



In [ ]:
# 定义神经网络
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1)
        )
        
        # Xavier 初始化
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, t):
        return self.net(t)

model = PINN()
print(model)
print(f"总参数量: {sum(p.numel() for p in model.parameters())}")



---
## 4. 构造损失函数

本例中，网络需要满足两个约束：

1. 区间内部满足微分方程 $u'(t)+u(t)=0$；
2. 初始点满足 $u(0)=1$。

第一个约束需要计算网络输出对输入 $t$ 的导数。PyTorch 中可使用 `torch.autograd.grad` 进行自动微分。




In [ ]:
# 定义损失函数
def loss_function(model, t_pde, t_bc, u_bc):
    # t_pde: 区间内部采样点，用于计算方程残差
    # t_bc: 初始/边界点，本例中为 t=0
    # u_bc: 初始/边界值，本例中为 u(0)=1
    
    t_pde.requires_grad_(True)
    u_pred = model(t_pde)
    
    # 自动微分计算 du/dt
    du_dt = torch.autograd.grad(
        outputs=u_pred,
        inputs=t_pde,
        grad_outputs=torch.ones_like(u_pred),
        create_graph=True
    )[0]
    
    # 方程残差：du/dt + u = 0
    pde_residual = du_dt + u_pred
    loss_pde = torch.mean(pde_residual ** 2)
    
    # 初始条件损失
    u_bc_pred = model(t_bc)
    loss_bc = torch.mean((u_bc_pred - u_bc) ** 2)
    
    # 总损失
    lambda_bc = 10.0
    total_loss = loss_pde + lambda_bc * loss_bc
    
    return total_loss, loss_pde.item(), loss_bc.item()



---
## 5. 准备训练点

这里的训练点不同于普通监督学习中的标注数据。求解该方程不需要预先给出大量真实的 $u(t)$。

本例只需要两类点：

- 区间 $[0,3]$ 内的采样点，用于计算方程残差；
- 初始点 $t=0$，用于约束 $u(0)=1$。

也就是说，网络主要由微分方程和初始条件约束，而不是由大量标签数据约束。




In [ ]:
# 定义区间
t_min, t_max = 0.0, 3.0

# PDE 训练点：区间内随机采样
N_pde = 200
t_pde = torch.rand(N_pde, 1) * (t_max - t_min) + t_min

# 初始条件：t = 0, u = 1
t_bc = torch.tensor([[0.0]])
u_bc = torch.tensor([[1.0]])

print(f"PDE 内部点: {t_pde.shape}, 范围 [{t_pde.min().item():.2f}, {t_pde.max().item():.2f}]")
print(f"初始点: t={t_bc.item()}, u={u_bc.item()}")



---
## 6. 训练循环

下面使用 Adam 优化器训练网络。每轮训练包括三步：

1. 计算当前网络的总损失；
2. 反向传播，计算参数梯度；
3. 更新参数，降低损失函数。

训练过程中记录 PDE 损失和初始条件损失，用于检查收敛情况。




In [ ]:
# 定义优化器。这里使用 Adam 优化算法，学习率设为 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 训练轮数
n_epochs = 3000

# 用于记录每一轮训练中的 PDE 残差损失和初始条件损失
loss_history = []

# 开始训练
for epoch in range(n_epochs):
    
    # 清空上一轮反向传播累积的梯度
    optimizer.zero_grad()
    
    # 计算总损失、PDE 残差损失和初始条件损失
    total_loss, loss_pde, loss_bc = loss_function(model, t_pde, t_bc, u_bc)
    
    # 反向传播，计算损失函数对网络参数的梯度
    total_loss.backward()
    
    # 根据梯度更新神经网络参数
    optimizer.step()
    
    # 记录当前轮次的两个损失项
    loss_history.append([loss_pde, loss_bc])
    
    # 每 500 轮输出一次训练信息
    if (epoch + 1) % 500 == 0:
        print(
            f"Epoch {epoch+1:4d}/{n_epochs} | "
            f"total: {total_loss.item():.6e} | "
            f"PDE: {loss_pde:.6e} | "
            f"IC: {loss_bc:.6e}"
        )

print("\n训练结束")

---
## 7. 可视化结果

训练结束后绘制两张图：损失曲线，以及 PINN 预测结果与解析解的对比。




In [ ]:
# 损失曲线
# 将 loss_history 转为 numpy 数组，便于后续画图
loss_history = np.array(loss_history)

# 创建画布：左图显示损失函数变化，右图显示预测结果
plt.figure(figsize=(12, 4))

# =========================
# 左图：训练损失曲线
# =========================
plt.subplot(1, 2, 1)

# PDE 残差损失
plt.semilogy(loss_history[:, 0], label='PDE Loss')

# 初始条件损失
plt.semilogy(loss_history[:, 1], label='IC Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.legend()
plt.title('Training loss')
plt.grid(True)

# =========================
# 右图：PINN 预测结果与解析解对比
# =========================

# 构造测试点
t_test = torch.linspace(t_min, t_max, 100).reshape(-1, 1)

# 用训练好的网络预测 u(t)
u_pred = model(t_test).detach().numpy()

# 解析解：u(t) = exp(-t)
u_exact = np.exp(-t_test.numpy())

plt.subplot(1, 2, 2)

# 解析解
plt.plot(t_test.numpy(), u_exact, 'b-', label='Analytical solution $e^{-t}$', linewidth=2)

# PINN 预测结果
plt.plot(t_test.numpy(), u_pred, 'r--', label='PINN prediction', linewidth=2)

# PDE 训练点的位置。这里只是标出 t 的分布，所以纵坐标放在 0 附近
t_pde_np = t_pde.detach().numpy()
plt.scatter(t_pde_np, np.zeros_like(t_pde_np), s=5, alpha=0.3, label='PDE training points')

# 初始条件点
plt.scatter(
    t_bc.numpy(),
    u_bc.numpy(),
    s=80,
    c='g',
    marker='*',
    label='Initial condition',
    zorder=5
)

plt.xlabel('t')
plt.ylabel('u(t)')
plt.legend()
plt.title('PINN prediction vs analytical solution')
plt.grid(True)

plt.tight_layout()
plt.show()



---
## 8. 定量评估

除图像比较外，可进一步计算误差指标。这里给出平均相对误差和最大绝对误差。




In [ ]:
u_pred_np = u_pred.flatten()
u_exact_np = u_exact.flatten()

relative_error = np.mean(np.abs(u_pred_np - u_exact_np) / np.abs(u_exact_np)) * 100
max_error = np.max(np.abs(u_pred_np - u_exact_np))

print(f"平均相对误差: {relative_error:.2f}%")
print(f"最大绝对误差: {max_error:.6f}")
print(f"\n参数量: {sum(p.numel() for p in model.parameters())}")



---
## 9. 扩展：二阶 ODE

将问题改为二阶方程：

$$
\frac{d^2u}{dt^2} + u = 0, \quad t \in [0, 2\pi].
$$

为得到确定的非零解，设定初始条件：

$$
u(0)=0, \quad u'(0)=1.
$$

对应解析解为：

$$
u(t)=\sin t.
$$

该练习与一阶方程基本一致。区别在于需要先用 `autograd.grad` 计算 $du/dt$，再对 $du/dt$ 再次求导，得到 $d^2u/dt^2$。






---
## 10. PINN 的优点和局限

### 优点

- **不依赖固定网格**：采样点设置较灵活，对高维问题或复杂区域有一定吸引力。
- **输出连续函数**：训练后可在任意输入点得到预测结果。
- **适合反问题**：未知物理参数可以作为待训练变量。
- **便于融合数据和方程**：观测数据和物理残差可同时写入损失函数。
- **可加入先验约束**：例如对称性、边界行为、守恒关系等。

### 局限

- **训练效率不一定高**：对许多标准正问题，传统数值方法通常更高效。
- **对超参数敏感**：网络层数、神经元数、学习率和损失权重都会影响结果。
- **高精度求解较困难**：若问题要求高精度，PINN 未必是合适选择。
- **复杂多尺度问题训练困难**：强非线性、多尺度、激波或边界层问题可能出现收敛困难。

### 小结

PINN 可以概括为：用神经网络表示解，用物理方程约束解。

本例的基本流程为：

1. 用神经网络表示未知函数；
2. 用自动微分计算导数；
3. 根据微分方程写出残差；
4. 将方程残差和初始条件组成损失函数；
5. 通过优化器训练网络；
6. 用解析解检查结果。

Burgers 方程、扩散方程、Parker 太阳风方程等问题也可沿用这一流程。实际区别在于变量维度、方程形式和损失项设置更复杂。




---
## 附录：完整代码

以下为一阶 ODE 算例的完整代码。




In [ ]:
# PINN 完整示例：求解 u'(t) = -u(t), u(0) = 1
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 20), nn.Tanh(),
            nn.Linear(20, 20), nn.Tanh(),
            nn.Linear(20, 20), nn.Tanh(),
            nn.Linear(20, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, t):
        return self.net(t)

def loss_function(model, t_pde, t_bc, u_bc):
    t_pde.requires_grad_(True)
    u_pred = model(t_pde)
    du_dt = torch.autograd.grad(
        u_pred, t_pde,
        grad_outputs=torch.ones_like(u_pred),
        create_graph=True
    )[0]
    loss_pde = torch.mean((du_dt + u_pred) ** 2)
    loss_bc = torch.mean((model(t_bc) - u_bc) ** 2)
    return loss_pde + 10.0 * loss_bc

t_pde = torch.rand(200, 1) * 3.0
t_bc, u_bc = torch.tensor([[0.0]]), torch.tensor([[1.0]])

model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(3000):
    optimizer.zero_grad()
    loss = loss_function(model, t_pde, t_bc, u_bc)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.6e}")

t_test = torch.linspace(0, 3, 100).reshape(-1, 1)
u_pred = model(t_test).detach().numpy()
u_exact = np.exp(-t_test.numpy())

plt.plot(t_test.numpy(), u_exact, 'b-', label='Exact')
plt.plot(t_test.numpy(), u_pred, 'r--', label='PINN')
plt.legend()
plt.show()